# *Fine-Tuning* Lanjutan Model mT5 dengan Data Baru

Notebook ini memperbarui model yang sudah ada di Hugging Face (`Brnalxz/mt5-small-normalisasi-slang-game`) dengan data baru, misalnya slang yang baru ditambahkan ke kamus beserta contoh kalimatnya.

Alur notebook:

1. **Memuat data lama** (data latih, validasi, dan uji yang dipakai pada pelatihan pertama).
2. **Menyiapkan data baru**: dibersihkan, dibagi 80:10:10, lalu data latihnya diaugmentasi dengan cara yang sama.
3. **Pelatihan lanjutan** dari model lama, dengan data lama + data baru, supaya model mengenal slang baru tanpa melupakan yang lama.
4. **Evaluasi**: model lama dan model baru dibandingkan pada data uji lama dan data uji baru.
5. **Unggah ke Hugging Face** jika model baru lebih baik.
6. **Memakai model baru di aplikasi** tanpa mengubah kode.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
!pip install -q transformers datasets sentencepiece accelerate sacrebleu rouge_score bert_score huggingface_hub

In [ ]:
import glob
import json
import math
import random
import re
import string
from typing import NamedTuple

import bert_score
import matplotlib.pyplot as plt
import pandas as pd
import sacrebleu
import torch
from datasets import Dataset
from rouge_score import rouge_scorer
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    MT5ForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

## Konfigurasi

Unduh `data_baru.csv` dan `Kamus_Slang_Game.xlsx` dari **Panel Admin** aplikasi, lalu letakkan di folder Dataset. Dua kolom yang dipakai sama seperti *parallel corpus* (kolom `istilah` dan `ditambahkan` diabaikan):

| clean_comment | baku_reference |
|---|---|
| aduh dc lagi pas war | aduh putus koneksi lagi saat bertempur |
| masuk dc yuk biar enak koordinasi | masuk discord yuk supaya mudah berkoordinasi |

Untuk setiap slang baru, tulis 3–5 contoh kalimat. Untuk kata ambigu seperti "dc", buat contoh untuk setiap artinya. Pakai kamus terbaru dari Panel Admin di `DICTIONARY_FILE`.

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Dataset"
MODEL_DIR = "/content/drive/MyDrive/Model_mT5"

OLD_MODEL_ID = "Brnalxz/mt5-small-normalisasi-slang-game"

# Ubah bagian NEW_MODEL_ID jika ingin menyimpan model baru dengan nama berbeda di Hugging Face Hub
NEW_MODEL_ID = "Brnalxz/mt5-small-normalisasi-slang-game" 

OLD_TRAIN_FILE = f"{DATA_DIR}/train_augmented.csv"
OLD_VALID_FILE = f"{DATA_DIR}/valid.csv"
OLD_TEST_FILE = f"{DATA_DIR}/test.csv"
NEW_DATA_FILE = f"{DATA_DIR}/data_baru.csv"
DICTIONARY_FILE = f"{DATA_DIR}/Kamus_Slang_Game.xlsx"

CHECKPOINT_DIR = f"{MODEL_DIR}/checkpoint_lanjutan"
FINAL_MODEL_DIR = f"{MODEL_DIR}/mt5_slang_game_lanjutan"

SOURCE = "clean_comment"
TARGET = "baku_reference"
PREFIX = "normalisasi teks game ke bahasa Indonesia baku: "
MAX_LENGTH = 64
NUM_BEAMS = 4

# Model sudah terlatih, jadi learning rate dibuat lebih kecil dan epoch lebih sedikit
# daripada pelatihan pertama (5e-4, 10 epoch) agar pengetahuan lama tidak rusak.
LEARNING_RATE = 1e-4
NUM_EPOCHS = 5

SEED = 42
VARIANTS_PER_SENTENCE = 4

TRAIN_MODEL = True
PUSH_TO_HUB = False   # ubah ke True setelah hasil evaluasi diperiksa

random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Perangkat:", device)

## 1. Memuat data lama

In [ ]:
def load_split(path):
    df = pd.read_csv(path)
    return df.dropna(subset=[SOURCE, TARGET])[[SOURCE, TARGET]].reset_index(drop=True)


old_train_df = load_split(OLD_TRAIN_FILE)
old_valid_df = load_split(OLD_VALID_FILE)
old_test_df = load_split(OLD_TEST_FILE)

print(f"Data lama  latih: {len(old_train_df)} | validasi: {len(old_valid_df)} | uji: {len(old_test_df)}")

## 2. Menyiapkan data baru

Kalimat masukan dibersihkan dengan aturan yang sama dengan `preprocess()` di `App.py`. Data baru dibagi 80:10:10 **sebelum** augmentasi, lalu pasangan yang sudah ada di data lama dibuang supaya tidak terjadi kebocoran data.

In [ ]:
EMOJI_RE = re.compile("[\U0001F1E6-\U0001F1FF\U0001F300-\U0001FAFF☀-➿⬀-⯿️‍]+")
URL_RE = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_RE = re.compile(r"[@#]\w+")
REPEATED_LETTER_RE = re.compile(r"([a-z])\1{2,}")
NON_TEXT_RE = re.compile(rf"[^a-z0-9\s{re.escape(string.punctuation)}]")
PUNCT_BEFORE_WORD_RE = re.compile(r"([.,!?;:])(?=[a-z])")
WHITESPACE_RE = re.compile(r"\s+")


def preprocess(text):
    # Sama dengan preprocess() di App.py.
    text = "" if pd.isna(text) else str(text).lower()
    text = URL_RE.sub(" ", text)
    text = MENTION_RE.sub(" ", text)
    text = EMOJI_RE.sub(" ", text)
    text = REPEATED_LETTER_RE.sub(r"\1", text)
    text = NON_TEXT_RE.sub(" ", text)
    text = PUNCT_BEFORE_WORD_RE.sub(r"\1 ", text)
    return WHITESPACE_RE.sub(" ", text).strip()


new_df = pd.read_csv(NEW_DATA_FILE)[[SOURCE, TARGET]].dropna()
new_df[SOURCE] = new_df[SOURCE].apply(preprocess)
new_df[TARGET] = new_df[TARGET].astype(str).str.strip()
new_df = new_df[(new_df[SOURCE] != "") & (new_df[TARGET] != "")]
new_df = new_df.drop_duplicates(subset=[SOURCE, TARGET])

old_sources = set(old_train_df[SOURCE]) | set(old_valid_df[SOURCE]) | set(old_test_df[SOURCE])
new_df = new_df[~new_df[SOURCE].isin(old_sources)].reset_index(drop=True)
print("Pasangan kalimat baru:", len(new_df))

if len(new_df) >= 10:
    new_train_df, holdout_df = train_test_split(new_df, test_size=0.2, random_state=SEED)
    new_valid_df, new_test_df = train_test_split(holdout_df, test_size=0.5, random_state=SEED)
else:
    print("Data baru kurang dari 10 pasangan: semuanya dipakai untuk latih, tanpa data uji baru.")
    new_train_df, new_valid_df, new_test_df = new_df, new_df.iloc[0:0], new_df.iloc[0:0]

print(f"Data baru  latih: {len(new_train_df)} | validasi: {len(new_valid_df)} | uji: {len(new_test_df)}")

Augmentasi data latih baru memakai aturan yang sama dengan notebook augmentasi.

In [ ]:
WORD_VARIATIONS = {
    "bang": ["bang", "bg", "bro"],
    "bro": ["bro", "bang", "bg"],
    "gua": ["gua", "gw", "gue"],
    "gw": ["gw", "gua", "gue"],
    "gue": ["gue", "gw", "gua"],
    "saya": ["saya", "aku"],
    "tidak": ["tidak", "ga", "gak", "g"],
    "ga": ["ga", "gak", "g", "tidak"],
    "gak": ["gak", "ga", "g", "tidak"],
    "aja": ["aja", "saja"],
    "udah": ["udah", "sudah"],
    "blm": ["blm", "belum"],
    "bgt": ["bgt", "banget", "sekali"],
    "kalo": ["kalo", "kalau"],
    "tp": ["tp", "tapi"],
    "yg": ["yg", "yang"],
    "dpt": ["dpt", "dapet", "mendapat"],
    "lvl": ["lvl", "level"],
    "ult": ["ult", "ultimate"],
    "exp": ["exp", "experience"],
    "mm": ["mm", "marksman"],
    "jg": ["jg", "juga"],
}
EXTRA_WORDS = ["", "", "", " nih", " cuy", " bro", " bang"]
MAX_TRIES = 50


def augment_sentence(sentence):
    words = [
        random.choice(WORD_VARIATIONS[word.lower()]) if word.lower() in WORD_VARIATIONS else word
        for word in sentence.split()
    ]
    variant = " ".join(words) + random.choice(EXTRA_WORDS)
    return WHITESPACE_RE.sub(" ", variant).strip()


def generate_variants(sentence, count=VARIANTS_PER_SENTENCE):
    """Variasi unik yang berbeda dari kalimat asli; berhenti jika variasi tidak cukup."""
    variants = []
    for _ in range(MAX_TRIES):
        if len(variants) == count:
            break
        candidate = augment_sentence(sentence)
        if candidate != sentence and candidate not in variants:
            variants.append(candidate)
    return variants


def augment_dataframe(df):
    rows = []
    for source, target in zip(df[SOURCE], df[TARGET]):
        rows.append({SOURCE: source, TARGET: target})
        for variant in generate_variants(source):
            rows.append({SOURCE: variant, TARGET: target})
    augmented_df = pd.DataFrame(rows, columns=[SOURCE, TARGET]).drop_duplicates()
    return augmented_df.sample(frac=1, random_state=SEED).reset_index(drop=True)


new_train_augmented_df = augment_dataframe(new_train_df)
print("Data latih baru sebelum augmentasi:", len(new_train_df))
print("Data latih baru setelah augmentasi:", len(new_train_augmented_df))

Data latih lama tetap diikutkan. Jika hanya data baru yang dipakai, model cenderung melupakan pola lama (*catastrophic forgetting*).

In [ ]:
train_df = pd.concat([old_train_df, new_train_augmented_df]).drop_duplicates().sample(frac=1, random_state=SEED).reset_index(drop=True)
valid_df = pd.concat([old_valid_df, new_valid_df]).drop_duplicates().reset_index(drop=True)

leaked = (set(train_df[SOURCE]) & (set(old_test_df[SOURCE]) | set(new_test_df[SOURCE])))
print(f"Data gabungan  latih: {len(train_df)} | validasi: {len(valid_df)}")
print("Kalimat uji yang ada di data latih:", len(leaked))

## 3. Pelatihan lanjutan

Pelatihan dimulai dari model lama di Hugging Face, bukan dari `google/mt5-small`. Pengaturan lain sama dengan pelatihan pertama, dan *checkpoint* dengan *validation loss* terendah disimpan sebagai model akhir.

In [ ]:
def tokenize_dataset(df, tokenizer):
    def tokenize_batch(batch):
        inputs = [PREFIX + str(text) for text in batch[SOURCE]]
        targets = [str(text) for text in batch[TARGET]]
        encoded = tokenizer(inputs, max_length=MAX_LENGTH, truncation=True)
        encoded["labels"] = tokenizer(text_target=targets, max_length=MAX_LENGTH, truncation=True)["input_ids"]
        return encoded

    dataset = Dataset.from_pandas(df)
    return dataset.map(tokenize_batch, batched=True, remove_columns=dataset.column_names)


def train_model():
    tokenizer = AutoTokenizer.from_pretrained(OLD_MODEL_ID)
    model = MT5ForConditionalGeneration.from_pretrained(OLD_MODEL_ID)

    training_args = Seq2SeqTrainingArguments(
        output_dir=CHECKPOINT_DIR,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=2,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        predict_with_generate=True,
        fp16=False,
        logging_steps=50,
        save_total_limit=1,
        report_to="none",
        seed=SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenize_dataset(train_df, tokenizer),
        eval_dataset=tokenize_dataset(valid_df, tokenizer),
        processing_class=tokenizer,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
    )
    trainer.train()
    trainer.save_model(FINAL_MODEL_DIR)
    tokenizer.save_pretrained(FINAL_MODEL_DIR)


if TRAIN_MODEL:
    train_model()

In [ ]:
def load_log_history(checkpoint_dir):
    state_files = glob.glob(f"{checkpoint_dir}/checkpoint-*/trainer_state.json")
    if not state_files:
        raise FileNotFoundError(f"Tidak ada checkpoint pelatihan di {checkpoint_dir}")
    latest = max(state_files, key=lambda path: int(re.search(r"checkpoint-(\d+)", path).group(1)))
    with open(latest) as file:
        state = json.load(file)
    print("Checkpoint terbaik:", state["best_model_checkpoint"])
    return state["log_history"]


def loss_per_epoch(log_history):
    train_loss, valid_loss = {}, {}
    for entry in log_history:
        if "loss" in entry:
            train_loss[math.ceil(entry["epoch"])] = entry["loss"]
        if "eval_loss" in entry:
            valid_loss[round(entry["epoch"])] = entry["eval_loss"]
    return pd.DataFrame({"Training Loss": train_loss, "Validation Loss": valid_loss}).rename_axis("Epoch")


loss_df = loss_per_epoch(load_log_history(CHECKPOINT_DIR))
best_epoch = loss_df["Validation Loss"].idxmin()

ax = loss_df.plot(marker="o", figsize=(8, 4.5))
ax.axvline(best_epoch, color="gray", linestyle="--", label=f"Validation loss terendah (epoch {best_epoch})")
ax.set_xticks(loss_df.index)
ax.set_ylabel("Loss")
ax.legend()
plt.show()

loss_df.round(4)

## 4. *Lexical mapping* (sama dengan aplikasi)

In [ ]:
URL_PATTERN = re.compile(r"(https?://\S+|www\.\S+)")
MENTION_PATTERN = re.compile(r"[@#]\w+")
EMOJI_PATTERN = EMOJI_RE
REPEATED_LETTER_PATTERN = re.compile(r"([a-z])\1{2,}")
NON_ALPHANUMERIC_PATTERN = re.compile(r"[^a-z0-9\s]")
WHITESPACE_PATTERN = re.compile(r"\s+")
EDGE_PUNCTUATION_PATTERN = re.compile(r"^(\W*)(.*?)(\W*)$")

MAX_PHRASE_WORDS = 4
SENTENCE_END = (".", "!", "?")


def to_lookup_key(text):
    text = "" if pd.isna(text) else str(text).lower()
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = EMOJI_PATTERN.sub(" ", text)
    text = REPEATED_LETTER_PATTERN.sub(r"\1", text)
    text = NON_ALPHANUMERIC_PATTERN.sub(" ", text)
    return WHITESPACE_PATTERN.sub(" ", text).strip()


def load_slang_dictionary(path):
    kamus = pd.read_excel(path, sheet_name=0).iloc[:, :2].dropna().astype(str)
    dictionary = {}
    for slang, baku in zip(kamus.iloc[:, 0], kamus.iloc[:, 1].str.strip()):
        key = to_lookup_key(slang)
        if key and baku and key not in dictionary:
            dictionary[key] = baku
    return dictionary


SLANG_DICTIONARY = load_slang_dictionary(DICTIONARY_FILE)
PHRASE_LIMIT = min(max(len(key.split()) for key in SLANG_DICTIONARY), MAX_PHRASE_WORDS)
print("Jumlah entri kamus:", len(SLANG_DICTIONARY))


class Token(NamedTuple):
    prefix: str
    word: str
    suffix: str
    key: str


def tokenize(text):
    tokens = []
    for raw in text.split():
        prefix, word, suffix = EDGE_PUNCTUATION_PATTERN.match(raw).groups()
        tokens.append(Token(prefix, word, suffix, to_lookup_key(word)))
    return tokens


def phrase_key(tokens):
    if any(t.suffix for t in tokens[:-1]) or any(t.prefix for t in tokens[1:]):
        return ""
    return " ".join(t.key for t in tokens if t.key)


def find_longest_match(tokens, start):
    for length in range(min(PHRASE_LIMIT, len(tokens) - start), 0, -1):
        key = phrase_key(tokens[start:start + length])
        if key in SLANG_DICTIONARY:
            return length, SLANG_DICTIONARY[key]
    return 0, None


def match_capitalization(original, replacement, at_sentence_start):
    if original[:1].isupper() and (at_sentence_start or not original.isupper()):
        return replacement[:1].upper() + replacement[1:]
    return replacement


def lexical_mapping(text):
    tokens = tokenize(str(text))
    output, i = [], 0
    while i < len(tokens):
        length, baku = find_longest_match(tokens, i)
        if length:
            at_sentence_start = i == 0 or tokens[i - 1].suffix.endswith(SENTENCE_END)
            baku = match_capitalization(tokens[i].word, baku, at_sentence_start)
            output.append(tokens[i].prefix + baku + tokens[i + length - 1].suffix)
            i += length
        else:
            output.append(tokens[i].prefix + tokens[i].word + tokens[i].suffix)
            i += 1
    return " ".join(output)

## 5. Evaluasi: model lama dan model baru

Kedua model dinilai dengan skenario mT5 dan mT5 + *lexical mapping* pada:
- **data uji lama**: memastikan model baru tidak menjadi lebih buruk pada data lama;
- **data uji baru**: melihat apakah model baru mengenal slang baru.

In [ ]:
ROUGE = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)


def compute_metrics(predictions, references):
    predictions = [str(p) for p in predictions]
    references = [str(r) for r in references]

    bleu = sacrebleu.corpus_bleu(predictions, [references]).score
    rouge_l = sum(ROUGE.score(ref, pred)["rougeL"].fmeasure
                  for pred, ref in zip(predictions, references)) / len(references) * 100
    precision, recall, f1 = bert_score.score(predictions, references,
                                             model_type="bert-base-multilingual-cased",
                                             batch_size=64, verbose=False)
    return {
        "BLEU": bleu,
        "ROUGE-L": rouge_l,
        "BERTScore Precision": precision.mean().item() * 100,
        "BERTScore Recall": recall.mean().item() * 100,
        "BERTScore F1": f1.mean().item() * 100,
    }


def load_model(path):
    tokenizer = AutoTokenizer.from_pretrained(path)
    model = MT5ForConditionalGeneration.from_pretrained(path).to(device).eval()
    return tokenizer, model


def normalize_with_mt5(texts, tokenizer, model, batch_size=32):
    results = []
    for start in tqdm(range(0, len(texts), batch_size), desc="mT5"):
        batch = [PREFIX + str(text) for text in texts[start:start + batch_size]]
        encoded = tokenizer(batch, return_tensors="pt", padding=True, truncation=True,
                            max_length=MAX_LENGTH).to(device)
        with torch.no_grad():
            output = model.generate(**encoded, max_length=MAX_LENGTH, num_beams=NUM_BEAMS, early_stopping=True)
        results.extend(tokenizer.batch_decode(output, skip_special_tokens=True))
    return results


MODELS = {"Model lama": OLD_MODEL_ID, "Model baru": FINAL_MODEL_DIR}
TEST_SETS = {"Data uji lama": old_test_df, "Data uji baru": new_test_df}

rows, outputs = [], {}
for model_name, path in MODELS.items():
    tokenizer, model = load_model(path)
    for test_name, test_df in TEST_SETS.items():
        if test_df.empty:
            continue
        sources = test_df[SOURCE].tolist()
        references = test_df[TARGET].tolist()
        mt5_output = normalize_with_mt5(sources, tokenizer, model)
        final_output = [lexical_mapping(text) for text in mt5_output]
        outputs[(model_name, test_name)] = final_output
        for scenario, predictions in [("mT5", mt5_output), ("mT5 + Lexical Mapping", final_output)]:
            rows.append({"Data uji": test_name, "Skenario": scenario, "Model": model_name,
                         **compute_metrics(predictions, references)})
    del model
    torch.cuda.empty_cache()

results = pd.DataFrame(rows).set_index(["Data uji", "Skenario", "Model"]).round(2)
results

In [ ]:
comparison = results.unstack("Model")
for metric in ["BLEU", "ROUGE-L", "BERTScore F1"]:
    comparison[(metric, "Perubahan")] = comparison[(metric, "Model baru")] - comparison[(metric, "Model lama")]
comparison = comparison[[(m, c) for m in ["BLEU", "ROUGE-L", "BERTScore F1"] for c in ["Model lama", "Model baru", "Perubahan"]]]
comparison.round(2)

In [ ]:
if not new_test_df.empty:
    example_df = pd.DataFrame({
        "Masukan": new_test_df[SOURCE].tolist(),
        "Referensi": new_test_df[TARGET].tolist(),
        "Model lama": outputs[("Model lama", "Data uji baru")],
        "Model baru": outputs[("Model baru", "Data uji baru")],
    })
    pd.set_option("display.max_colwidth", None)
    display(example_df)

## 6. Mengunggah model baru ke Hugging Face

Unggah hanya jika:
- skor pada **data uji lama** tidak turun jauh (model tidak melupakan pola lama), dan
- skor pada **data uji baru** naik dibanding model lama.

Simpan token Hugging Face (izin *write*) di menu **Secrets** Colab (ikon kunci) dengan nama `HF_TOKEN`. Jika `NEW_MODEL_ID` sama dengan model lama, model baru menjadi versi (*commit*) terbaru di repo yang sama, dan versi lama tetap tersimpan di riwayatnya.

In [ ]:
if PUSH_TO_HUB:
    from google.colab import userdata
    from huggingface_hub import login

    login(token=userdata.get("HF_TOKEN"))
    tokenizer, model = load_model(FINAL_MODEL_DIR)
    message = f"Fine-tuning lanjutan: +{len(new_df)} pasangan kalimat baru"
    commit = model.push_to_hub(NEW_MODEL_ID, commit_message=message)
    tokenizer.push_to_hub(NEW_MODEL_ID, commit_message=message)
    print("Model diunggah ke:", NEW_MODEL_ID)
    print("Commit:", getattr(commit, "oid", commit))

## 7. Memakai model baru di aplikasi

Kode aplikasi tidak perlu diubah. Buka **Panel Admin** → tab **Pembaruan Model**:

1. Jika model baru diunggah ke **repo yang sama**: klik **Muat Ulang Model**, aplikasi memuat versi terbaru (`main`).
2. Jika diunggah ke **repo baru**: isi Model ID baru di **Ganti Model**, lalu klik **Simpan & Muat Ulang**.
3. Untuk kembali ke versi lama: isi Versi dengan kode *commit* versi lama dari tab *Files and versions* di Hugging Face.